In [2]:
# Step 1: upgrade pip
%pip install --upgrade pip

# Step 2: PyTorch + CUDA wheel
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

# Step 3: Hugging Face + helpers
%pip install transformers datasets evaluate scikit-learn accelerate tqdm

Note: you may need to restart the kernel to use updated packages.
Looking in indexes: https://download.pytorch.org/whl/cu118
Note: you may need to restart the kernel to use updated packages.
Note: you may need to restart the kernel to use updated packages.


In [1]:
#cell2
# check GPU / CUDA availability
import torch, sys
print("torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("CUDA version (torch):", torch.version.cuda)
print("GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    try:
        print("GPU name:", torch.cuda.get_device_name(0))
    except Exception as e:
        print("Couldn't get device name:", e)

# Optionally run nvidia-smi from the notebook (will only work if nvidia-smi is on PATH)
import subprocess, shlex
try:
    print("\n=== nvidia-smi output ===")
    print(subprocess.check_output(shlex.split("nvidia-smi")).decode())
except Exception as e:
    print("nvidia-smi not available or failed:", e)


torch: 2.7.1+cu118
CUDA available: True
CUDA version (torch): 11.8
GPU count: 1
GPU name: NVIDIA GeForce RTX 3050 Laptop GPU

=== nvidia-smi output ===
Thu Sep 25 23:24:31 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 566.07                 Driver Version: 566.07         CUDA Version: 12.7     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                  Driver-Model | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3050 ...  WDDM  |   00000000:01:00.0 Off |                  N/A |
| N/A   47C    P8              5W /   30W |     154MiB /   4096MiB |      0%    

In [2]:
#cell3
import os, random, numpy as np, pandas as pd
from datasets import Dataset
from sklearn.model_selection import train_test_split
import torch

# set random seed for reproducibility
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)


In [3]:
#cell4
import pandas as pd
import os

csv_path = "issue_detection_dataset.csv"
assert os.path.exists(csv_path), f"{csv_path} not found in current folder."

# read CSV, ensure first row is treated as header
df = pd.read_csv(csv_path, header=0)

# keep only first two columns (text, label)
df = df.iloc[:, :2].copy()
df.columns = ["text", "label"]

# drop rows with missing values
df = df.dropna(subset=["text", "label"]).reset_index(drop=True)

# drop any row where label literally equals 'label' (from header or bad row)
df = df[df['label'].str.lower() != 'label'].reset_index(drop=True)

print("Columns used:", df.columns.tolist())
print("Rows after cleaning:", len(df))
df.head(6)


Columns used: ['text', 'label']
Rows after cleaning: 14086


,text,label
0,The drainage system on MG Road is completely b...,issue
1,The drainage system on MG Road is completely b...,issue
2,The drainage system on MG Road is blocked comp...,issue
3,I think the new education policy will really h...,not_issue
4,I think the new education policy will really h...,not_issue
5,I think the new education policy testament rea...,not_issue


In [4]:
#cell5
# normalize label strings
df['label'] = df['label'].astype(str).str.strip().str.lower().str.replace(r'\W+', '_', regex=True)

# only keep expected labels
expected_labels = ['issue', 'not_issue']
df = df[df['label'].isin(expected_labels)].reset_index(drop=True)

# map labels to ids: issue=0, not_issue=1 (you can swap if you want the opposite)
label2id = {'issue': 0, 'not_issue': 1}
id2label = {v: k for k, v in label2id.items()}

# apply mapping
df['label'] = df['label'].map(label2id)

print("Final labels used:", df['label'].unique())
print("label2id mapping:", label2id)
df.head(6)


Final labels used: [0 1]
label2id mapping: {'issue': 0, 'not_issue': 1}


,text,label
0,The drainage system on MG Road is completely b...,0
1,The drainage system on MG Road is completely b...,0
2,The drainage system on MG Road is blocked comp...,0
3,I think the new education policy will really h...,1
4,I think the new education policy will really h...,1
5,I think the new education policy testament rea...,1


In [5]:
#cell6
from datasets import Dataset
hf_ds = Dataset.from_pandas(df)       # creates a Dataset with columns: text, label, and index
# drop the pandas index column if present
if 'index' in hf_ds.column_names:
    hf_ds = hf_ds.remove_columns('index')

# train / eval split
split = hf_ds.train_test_split(test_size=0.1, seed=42)
train_ds = split['train']
eval_ds  = split['test']
print("Train size:", len(train_ds), " Eval size:", len(eval_ds))


Train size: 12677  Eval size: 1409


In [6]:
#cell7
from transformers import AutoTokenizer

MODEL_NAME = "bert-base-uncased"   # you can change to another model if you like
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True, truncation=True)

# tokenization function
def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, padding=False, max_length=256)

# map tokenization (batched)
train_tok = train_ds.map(tokenize_batch, batched=True, remove_columns=["text"])
eval_tok  = eval_ds.map(tokenize_batch, batched=True, remove_columns=["text"])

# set the dataset format for PyTorch tensors (Trainer will handle conversion but this is convenient)
train_tok.set_format(type="torch")
eval_tok.set_format(type="torch")

print(train_tok.features)


Map:   0%|          | 0/12677 [00:00<?, ? examples/s]

Map:   0%|          | 0/1409 [00:00<?, ? examples/s]

{'label': Value('int64'), 'input_ids': List(Value('int32')), 'token_type_ids': List(Value('int8')), 'attention_mask': List(Value('int8'))}


In [7]:
#cell 8
from transformers import AutoModelForSequenceClassification, DataCollatorWithPadding

num_labels = len(label2id)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=num_labels)

# move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print("Model on device:", device)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model on device: cuda


In [10]:
%pip install --upgrade transformers

Note: you may need to restart the kernel to use updated packages.


In [12]:
import transformers
print(transformers.__version__)

4.56.2


In [14]:
import transformers
print(transformers.__file__)


C:\Users\rosha\AppData\Local\Programs\Python\Python313\Lib\site-packages\transformers\__init__.py


In [14]:
#cell9
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir="./bert_issue_detection",
    num_train_epochs=3,
    per_device_train_batch_size=8,   # older versions use per_device_train_batch_size
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    save_total_limit=2,
    seed=42,
)
from transformers import Trainer, DataCollatorWithPadding

# assuming model, tokenizer, train_tok, eval_tok are already defined
data_collator = DataCollatorWithPadding(tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=eval_tok,
    tokenizer=tokenizer,
    data_collator=data_collator,
)


C:\Users\rosha\AppData\Local\Temp\ipykernel_5356\1265040390.py:20: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [15]:
#cell 10
# Start training
trainer.train()


Step,Training Loss
500,0.070800
1000,0.015200
1500,0.002400
2000,0.001600
2500,0.000000
3000,0.000000
3500,0.001600
4000,0.002500
4500,0.000000


TrainOutput(global_step=4755, training_loss=0.009911327933026898, metrics={'train_runtime': 606.3994, 'train_samples_per_second': 62.716, 'train_steps_per_second': 7.841, 'total_flos': 599298123013560.0, 'train_loss': 0.009911327933026898, 'epoch': 3.0})

In [16]:
#cell11
# Save the fine-tuned model and tokenizer
model_save_path = "./best_bert_issue_detector"
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)

print(f"Model and tokenizer saved to: {model_save_path}")


Model and tokenizer saved to: ./best_bert_issue_detector


In [17]:
#cell12
# Evaluate on the evaluation dataset
metrics = trainer.evaluate(eval_dataset=eval_tok)

print("Evaluation metrics:")
for key, value in metrics.items():
    print(f"{key}: {value:.4f}")


Evaluation metrics:
eval_loss: 0.0008
eval_runtime: 2.1106
eval_samples_per_second: 667.5860
eval_steps_per_second: 42.1680
epoch: 3.0000


In [18]:
#cell13
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score
import torch
import numpy as np

# Get predictions
preds_output = trainer.predict(eval_tok)
logits = preds_output.predictions
labels = preds_output.label_ids

# Convert logits to predicted class indices
preds = np.argmax(logits, axis=1)

# Compute metrics
accuracy = accuracy_score(labels, preds)
f1 = f1_score(labels, preds, average="weighted")
precision = precision_score(labels, preds, average="weighted")
recall = recall_score(labels, preds, average="weighted")

print(f"Accuracy: {accuracy:.4f}")
print(f"F1 Score: {f1:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")


Accuracy: 0.9993
F1 Score: 0.9993
Precision: 0.9993
Recall: 0.9993


In [7]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Load saved model and tokenizer
model_path = "./best_bert_issue_detector"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.eval()

# Example texts to predict
texts = [
    "Road is blocked in Bhopal.",
    "Rahul gandhi should be next PM."
]

# Tokenize
enc = tokenizer(texts, truncation=True, padding=True, max_length=256, return_tensors="pt").to(device)

# Predict
with torch.no_grad():
    logits = model(**enc).logits
    preds = torch.argmax(logits, dim=1).cpu().numpy()

# Map predictions back to labels
id2label = {0: "issue", 1: "not_issue"}  # make sure this matches your label2id mapping
pred_labels = [id2label[p] for p in preds]

for t, l in zip(texts, pred_labels):
    print(f"Text: {t}\nPredicted label: {l}\n")


Text: Road is blocked in Bhopal.
Predicted label: issue

Text: Rahul gandhi should be next PM.
Predicted label: not_issue

